# Chapter 14 &mdash; High-Level Proof Sketches, and the Strictness of the Hierarchy

**Concept 12 of the Chapter 14 decomposition:** *High-Level Proof Sketches, and the Strictness of the Hierarchy*

A worked catalogue: $L_{UnivDFA}$, CFG emptiness and LBA halting are recursive &mdash; and each level is strict.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-High-Level-Proof-Sketches/Concept-High-Level-Proof-Sketches.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A catalogue of decidability results, each with a one-paragraph reason:

* **$L_{UnivDFA} = \{\langle D\rangle : L(D)=\Sigma^*\}$** &mdash; recursive.
  Complement the DFA and test emptiness (Concept 5).
* **CFG emptiness** &mdash; recursive. Mark the productive nonterminals by a least fixed
  point (Chapter 11, Concept 5) and ask whether $S$ is marked.
* **LBA halting** &mdash; recursive. An LBA has finitely many configurations on a tape of
  length $n$ (at most $|Q|\cdot n\cdot|\Gamma|^n$), so run it that many steps; if it has
  not halted it is in a loop.
* **DFA equivalence** &mdash; recursive (Chapter 6, lock-step search).
* **CFG equivalence** &mdash; **not** recursive (Concept 8).

The common shape: **decidable when the state space is finite and computable; undecidable
when it is not.**

## 2. Definitions

### Three deciders

In [ ]:
def decide_empty_dfa(D):
    seen, frontier = {D["q0"]}, {D["q0"]}
    while frontier:
        frontier = {step_dfa(D, q, a) for q in frontier for a in D["Sigma"]} - seen
        seen |= frontier
    return not (seen & D["F"])

def decide_universal_dfa(D):
    return decide_empty_dfa(comp_dfa(D))       # L(D) = Sigma*  iff  complement is empty

### CFG emptiness, and the LBA configuration count

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps


def decide_empty_cfg(G):
    productive, changed = set(), True
    while changed:
        changed = False
        for A, rhss in G['P'].items():
            if A in productive: continue
            if any(all(x in G['Sigma'] or x in productive for x in r) for r in rhss):
                productive.add(A); changed = True
    return G['S'] not in productive

def lba_config_bound(nQ, n, nGamma):
    return nQ * max(n, 1) * (nGamma ** max(n, 1))

## 3. Tests

**$L_{UnivDFA}$** is decidable: complement, then test emptiness.

In [ ]:
Univ  = md2mc('DFA\nIF : 0 | 1 -> IF\n')
NotU  = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> D
D  : 0 | 1 -> D
''')
print("L(Univ) == Sigma* ?", decide_universal_dfa(Univ))
print("L(NotU) == Sigma* ?", decide_universal_dfa(NotU))
assert decide_universal_dfa(Univ) and not decide_universal_dfa(NotU)

**CFG emptiness** is decidable: a least fixed point over the nonterminals.

In [ ]:
Good   = mkg({'S': ["", "aSb"]})
Barren = mkg({'S': ["aSb", "SS"]})          # no terminal-only right-hand side
Partly = mkg({'S': ["aA", "b"], 'A': ["aA"]})
for name, G in [('Good', Good), ('Barren', Barren), ('Partly', Partly)]:
    print("  %-8s L(G) empty? %-6s  (language up to 4: %s)"
          % (name, decide_empty_cfg(G), language(G, 4)[:4]))
assert not decide_empty_cfg(Good) and decide_empty_cfg(Barren)
assert not decide_empty_cfg(Partly)

**LBA halting** is decidable because the configuration space is finite.

In [ ]:
print("%-6s %-10s %s" % ("n", "|configs|", "so run at most this many steps"))
for n in [1, 2, 4, 8]:
    b = lba_config_bound(nQ=5, n=n, nGamma=3)
    print("%-6d %-10d %d" % (n, b, b))
print()
print("Finite, so: run that many steps.  Still going? Then it is in a loop.")
print("Astronomically large, but FINITE -- decidability is not about speed.")

The pattern, stated.

In [ ]:
CAT = [("L_EmptyDFA",    "recursive",     "finite graph reachability"),
       ("L_UnivDFA",     "recursive",     "complement + emptiness"),
       ("DFA equivalence","recursive",    "lock-step search, Chapter 6"),
       ("CFG emptiness",  "recursive",    "least fixed point over nonterminals"),
       ("LBA halting",    "recursive",    "finite configuration space"),
       ("CFG equivalence","NOT recursive","behaviour, unbounded"),
       ("A_TM",           "RE only",      "behaviour, unbounded"),
       ("complement A_TM","not even RE",  "Concept 7 contrapositive")]
print("%-18s %-16s %s" % ("problem", "status", "why"))
for a, b, d in CAT:
    print("%-18s %-16s %s" % (a, b, d))

And the strictness of every level, with its witness.

In [ ]:
STRICT = [("regular < CFL",        "a^n b^n"),
          ("CFL < CSL",            "a^n b^n c^n"),
          ("CSL < recursive",      "a diagonal language over LBAs"),
          ("recursive < RE",       "A_TM"),
          ("RE < all languages",   "complement of A_TM")]
for rel, wit in STRICT:
    print("  %-22s witness %s" % (rel, wit))
assert len(STRICT) == 5
print("\nEvery step of the hierarchy is strict, and every step has a name.")

## 4. Animation

A DFA whose language is $\Sigma^*$ &mdash; the easy case of $L_{UnivDFA}$.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(md2mc('DFA\nIF : 0 | 1 -> IF\n'), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write out the LBA-halting decider in pseudo-code. Where does the bound come from?
2. Why is CFG emptiness decidable but CFG equivalence not?
3. Which decidable problem in the catalogue has the worst running time?

In [ ]:
# Your work for the exercises above.